In [ ]:
# Cell 1: Setup and Data Preparation
# =========================================
# 🧩 Deepfake Detection with EfficientNet, SE, and Attention Modules
# Dataset: WildDeepfake (KaggleHub)
# =========================================

# Install required packages first
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
    except subprocess.CalledProcessError:
        print(f"❌ Failed to install {package}")

# Install kagglehub if not available
try:
    import kagglehub
    print("✅ kagglehub already installed")
except ImportError:
    print("📦 Installing kagglehub...")
    install_package("kagglehub")
    import kagglehub

# Standard imports
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import (GlobalAveragePooling2D, Dense, Dropout, Multiply, 
                                   Reshape, Conv2D, Add, Lambda, Concatenate, GlobalMaxPooling2D,
                                   BatchNormalization, Activation, Input)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

# Set memory growth for GPU if available
if tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(tf.config.list_physical_devices('GPU')[0], True)
        print("🎮 GPU memory growth enabled")
    except:
        print("💻 GPU available but memory growth not set")
else:
    print("💻 Running on CPU")

In [ ]:
# Cell 2: Dataset Download and Setup
# =========================================
# 1️⃣ Download and Setup Dataset
# =========================================

def download_and_setup_dataset():
    """Download and setup the WildDeepfake dataset with proper error handling"""
    try:
        print("⬇️ Downloading WildDeepfake dataset...")
        download_path = kagglehub.dataset_download("maysuni/wild-deepfake")
        print(f"✅ Dataset downloaded at: {download_path}")
        
        # Unzip if necessary
        dataset_path = download_path
        for file in os.listdir(download_path):
            if file.endswith(".zip"):
                zip_path = os.path.join(download_path, file)
                extract_dir = os.path.join(download_path, "WildDeepfake")
                if not os.path.exists(extract_dir):
                    print("📦 Extracting dataset...")
                    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                        zip_ref.extractall(extract_dir)
                    print("✅ Dataset extracted successfully")
                dataset_path = extract_dir
                break

        # Verify the path and find the correct directory containing 'fake' and 'real' folders
        if not os.path.exists(os.path.join(dataset_path, "fake")) or not os.path.exists(os.path.join(dataset_path, "real")):
            print("🔍 Searching for 'fake' and 'real' directories...")
            found_path = None
            for root, dirs, files in os.walk(download_path):
                if 'fake' in dirs and 'real' in dirs:
                    found_path = root
                    break
            if found_path:
                dataset_path = found_path
                print(f"✅ Found directories at: {dataset_path}")
            else:
                raise FileNotFoundError("❌ Could not find 'fake' and 'real' directories in dataset.")

        print(f"📁 Using dataset path: {dataset_path}")
        
        # Verify dataset contents
        fake_dir = os.path.join(dataset_path, "fake")
        real_dir = os.path.join(dataset_path, "real")
        
        if os.path.exists(fake_dir) and os.path.exists(real_dir):
            fake_count = len([f for f in os.listdir(fake_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            real_count = len([f for f in os.listdir(real_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            print(f"📊 Dataset verification:")
            print(f"   - Fake images: {fake_count}")
            print(f"   - Real images: {real_count}")
            print(f"   - Total images: {fake_count + real_count}")
        
        return dataset_path
        
    except Exception as e:
        print(f"❌ Error downloading dataset: {str(e)}")
        print("💡 Please manually download the dataset and update the path below")
        # Return a placeholder path - user should update this
        return r"C:\path\to\your\dataset"

# Download and setup dataset
dataset_path = download_and_setup_dataset()